# Delooping 

The previous script in Matlab (6_script_plot_loops) will return a text file listing all loops and the nodes involved, and a separate .dat file giving the xyz coordinates of the nodes. 
This notebook will take the two files, the input image, and the skeleton image, and delete skeleton pixels that are on the contour of the input image. Many loops come from fused tips or fused tip to branches. 
It will also prompt you to add the voxel size (7um up to PCW10, 4um for PCW10+), and the thickness of the contour that you want to use. Start with 5, and re-run 6_script_plot_loops. If there are still loops left, use 10, then 20. 
From my experience, most datasets will work after using 10. 

In [6]:
# Imports 
import pandas as pd 
import cv2
import tifffile as tf
import numpy as np
import re
from scipy.ndimage import label
from skimage.morphology import skeletonize
import skan 
import os

In [7]:
# Configure the path and specify your images
folder_path = r"F:\Megumi\Dropbox (DBOX-EQS1)\Megumi-Cadisha\Skeleton by Lobe\Left Upper Lobe\proximal_pruning\EH4326LU-P9(9.7)\\"
skeleton_img = tf.imread(folder_path + "skeleton.tif")
input_img = tf.imread(folder_path + "input.tif")
voxel = int(input("voxel:"))
thickness = int(input("thickness:"))

# Open the loops.txt file to read the list of nodes involved in loops, and print it. 
# The list is stored as separate loop groups, we create a simple list of nodes. 
with open(folder_path + 'loops.txt', 'r') as file:
    # Initialize an empty list to store the numbers
    loop_list = []
    # Iterate through each line in the file
    for line in file:
        # Use regular expression to find all numbers within square brackets in the line
        line_numbers = re.findall(r'\[(.*?)\]', line)
        # Split the found numbers and convert them to integers
        for num_str in line_numbers:
            loop_list.extend(map(int, num_str.split()))

# Print the final list of numbers
print(loop_list)

node_positions = pd.read_csv(folder_path + "node_positions.csv", delimiter = ",", names=["Node_ID", "Y", "X", "Z"])
#node_positions.set_axis(["Node_ID", "Y", "X", "Z"], axis=1, inplace=True)
# Sanity check 
print(node_positions.head())

# We will use this function to extract the node positions for a user-specified node 
def find_coordinates(node_positions, node):
    node_data = node_positions.loc[node_positions["Node_ID"] == int(node)]
    x, y, z = node_data.iloc[0][["X", "Y", "Z"]]
    return x, y, z

# Create a DataFrame for node positions
output_df = []

for node in loop_list:
    x, y, z = find_coordinates(node_positions, node)
    output_df.append([node, x, y, z])
    
# Creating the DataFrame
columns = ["ID", "X", "Y", "Z"]
output_df = pd.DataFrame(output_df, columns=columns)

# Save the DataFrame to a CSV file with header
output_df.to_csv(folder_path + 'loop_positions.csv', index=False, header=True)
print("Loops file is saved!")

voxel: 7
thickness: 5


[998, 1035, 1186, 1194, 1223, 1186, 1207, 1223, 1365, 1380, 1394, 1708, 1719, 1729, 1738, 1708, 1719, 1738]
   Node_ID       Y       X       Z
0        1  335.00  195.00  18.000
1        2  355.00  145.00  19.000
2        3  294.67  212.67  21.667
3        4  268.67  237.67  21.667
4        5  305.00  186.00  22.000
Loops file is saved!


In [8]:
########## PATH based pruning ##################################
######### 1. Extract xyz coordinates of node positions #########
# Load the loop_positions.csv from the previous cell 
loop_positions = pd.read_csv(folder_path + "loop_positions.csv") 
loop_positions.head()

######### 2. Convert skeleton to paths with skan  #########
# We need to match paths in the skeleton to the loop_positions 
# We use the package skan (skeleton-analysis) to do this
# We transform the skeleton_img to a skeleton object in skan with user-specified voxel sizes 
binary_skeleton = skeletonize(skeleton_img) 
skeleton = skan.Skeleton(binary_skeleton, spacing = voxel)

# Create a paths_table for the skeleton. This lists all branches (paths) in the skeleton, and their starting and ending coordinates which we will use later. 
# In skan, starting (or "source") coordinates for a path are stored as (image-coord-src-0, image-coord-src-1, image-coord-src-2) for (z, y, x) coordinates. 
# Ending (or "destination") coordinates for a path are stored as (image-coord-dst-0, image-coord-dst-1, image-coord-dst-2) for (z, y, x) coordinates.
# Since we don't specify a root node in skan and this is NOT treated as a network, we can assume that for any given branch, starting or ending nodes may be assigned arbitrarily, that is, they can be flipped. 
all_paths = [
    skeleton.path_coordinates(i)
    for i in range(skeleton.n_paths)
    ]
paths_table = skan.summarize(skeleton) 


######### 3. Match node positions to paths  #########
# Initialize an empty list to store path-ids
matched_path_ids = []
# skan stores xyz coordinates of the starting and end points of paths. with the node xyz coordinates, we look for the paths that are connected to it. 
# these coordinates are not exactly the same as the matlab node output (eg. the node coordinates from matlab may be 1 pixel or two off from the path coordinates).
# We set the tolerance to 1.5. We can imagine that for a given node, it is involved in 2 paths but in skan, it would only show 1 of the 2.
# Thus, we set the tolerance to a higher number so that we can get both paths involved. 
# From my tests, the optimal value is 1.5. See loops tolerance slides for more. 
tolerance = 1.5

# Iterate through each row of df1
for index, row in loop_positions.iterrows():
    # Extract XYZ values from the current row of df1
    XYZ_values_src = [row["X"], row["Y"], row["Z"]]
    XYZ_values_dst = [row["X"], row["Y"], row["Z"]]  # Assuming the same XYZ values for src and dst
    
    # Check if XYZ values match with any row in df2 (src coordinates)
    matching_rows_src = paths_table[((paths_table["image-coord-src-0"] - XYZ_values_src[2]).abs() <= tolerance) &
                             ((paths_table["image-coord-src-1"] - XYZ_values_src[1]).abs() <= tolerance) &
                             ((paths_table["image-coord-src-2"] - XYZ_values_src[0]).abs() <= tolerance)]
    
    # Check if XYZ values match with any row in df2 (dst coordinates)
    matching_rows_dst = paths_table[((paths_table["image-coord-dst-0"] - XYZ_values_dst[2]).abs() <= tolerance) &
                             ((paths_table["image-coord-dst-1"] - XYZ_values_dst[1]).abs() <= tolerance) &
                             ((paths_table["image-coord-dst-2"] - XYZ_values_dst[0]).abs() <= tolerance)]
    
    # If a match is found in either src or dst coordinates, append the corresponding path-id to matched_path_ids
    if not matching_rows_src.empty:
        matched_path_ids.extend(matching_rows_src.index.tolist())
    elif not matching_rows_dst.empty:
        matched_path_ids.extend(matching_rows_dst.index.tolist())

# Print the matched path-ids
#print("Matched path-ids:", matched_path_ids)

######### 4. Highlight loops in skeleton image  #########
loops_skel = np.zeros_like(binary_skeleton)

# Initialize coords_list
coords_list = []
for path_id in matched_path_ids: 
    coords_list.extend(skeleton.path_coordinates(int(path_id)))
    
#print(coords_list) 

columns = ['z', 'y', 'x']
loop_coords = pd.DataFrame(coords_list, columns = columns) 
print(loop_coords)

for index, row in loop_coords.iterrows():
    x, y, z = int(row['x']), int(row['y']), int(row['z'])
    loops_skel[z, y, x] = 255


# If you want to save the loops_paths as an image stack: 
tf.imwrite(folder_path + "loops_paths.ome.tif", loops_skel)

######### 5. Delete loop pixels in skeleton image  #########
contour_stack = np.zeros_like(input_img) 

# Process each slice and save the resulting image stack
z = skeleton_img.shape[0]
with tf.TiffWriter(os.path.join(folder_path, f"path_based_deloop_thickness{thickness}.tif")) as tif:
    for im in range(z):
        skeleton_slice = binary_skeleton[im, :, :]
        loops_slice = loops_skel[im, :, :]
        skeleton_new = np.copy(skeleton_slice)
        # Find contours in the mask slice
        mask_slice = input_img[im, :, :]
        contours, _ = cv2.findContours(mask_slice, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contour_image = np.zeros_like(skeleton_slice)
        contour_image = cv2.drawContours(contour_image, contours, -1, 255, thickness=thickness)
        contour_stack[im, :, :] = contour_image
        skeleton_new[(mask_slice == 0) & (skeleton_slice == 255)] = 0
        skeleton_new[(loops_slice == 255) & (contour_image == 255)] = 0
        filename1 = f"delooped_skeleton_{im}"
        tif.write(skeleton_new, description=filename1, metadata=None)

         z      y      x
0    169.0   75.0  399.0
1    169.0   74.0  398.0
2    170.0   73.0  397.0
3    171.0   73.0  398.0
4    169.0   75.0  399.0
..     ...    ...    ...
343  248.0  300.0  128.0
344  248.0  299.0  128.0
345  247.0  298.0  129.0
346  247.0  297.0  129.0
347  247.0  296.0  130.0

[348 rows x 3 columns]


# TO save both contour image stack AND delooped stack
import cv2
import os
import tifffile as tf
import numpy as np
import pandas as pd
from skimage.morphology import skeletonize
import skan

# Configure the path 
folder_path = r"F:\Megumi\Dropbox (DBOX-EQS1)\Figure_Images\EH3822LU skeletonization errors\Loops Explanation\\"

skeleton_img = tf.imread(folder_path + "EH3822LU_4um_skel.ome.tif")
input_img = tf.imread(folder_path + "EH3822LU_4um.tif")

# Read loop positions from CSV
loop_positions = pd.read_csv(folder_path + "loop_positions.csv")

# Convert skeleton to paths with skan
voxel = int(4)
binary_skeleton = skeletonize(skeleton_img)
skeleton = skan.Skeleton(binary_skeleton, spacing=voxel)
all_paths = [skeleton.path_coordinates(i) for i in range(skeleton.n_paths)]
paths_table = skan.summarize(skeleton)

# Match node positions to paths
matched_path_ids = []
tolerance = 2

for index, row in loop_positions.iterrows():
    XYZ_values_src = [row["X"], row["Y"], row["Z"]]
    XYZ_values_dst = [row["X"], row["Y"], row["Z"]]

    matching_rows_src = paths_table[((paths_table["image-coord-src-0"] - XYZ_values_src[2]).abs() <= tolerance) &
                                    ((paths_table["image-coord-src-1"] - XYZ_values_src[1]).abs() <= tolerance) &
                                    ((paths_table["image-coord-src-2"] - XYZ_values_src[0]).abs() <= tolerance)]

    matching_rows_dst = paths_table[((paths_table["image-coord-dst-0"] - XYZ_values_dst[2]).abs() <= tolerance) &
                                    ((paths_table["image-coord-dst-1"] - XYZ_values_dst[1]).abs() <= tolerance) &
                                    ((paths_table["image-coord-dst-2"] - XYZ_values_dst[0]).abs() <= tolerance)]

    if not matching_rows_src.empty:
        matched_path_ids.extend(matching_rows_src.index.tolist())
    elif not matching_rows_dst.empty:
        matched_path_ids.extend(matching_rows_dst.index.tolist())

# Highlight loops in skeleton image
loops_skel = np.zeros_like(binary_skeleton)
coords_list = []

for path_id in matched_path_ids:
    coords_list.extend(skeleton.path_coordinates(int(path_id)))

columns = ['z', 'y', 'x']
loop_coords = pd.DataFrame(coords_list, columns=columns)

for index, row in loop_coords.iterrows():
    x, y, z = int(row['x']), int(row['y']), int(row['z'])
    loops_skel[z, y, x] = 255

tf.imwrite(folder_path + "loops_paths.ome.tif", loops_skel)

# Delete loop pixels in skeleton image
contour_stack = np.zeros_like(input_img)
delooped_stack = np.zeros_like(input_img)

output_delooped_tif_path = os.path.join(folder_path, "delooped_stack.tif")
output_contour_tif_path = os.path.join(folder_path, "contour_stack.tif")

delooped_tif_writer = tf.TiffWriter(output_delooped_tif_path)
contour_tif_writer = tf.TiffWriter(output_contour_tif_path)

z = skeleton_img.shape[0]
for im in range(z):
    skeleton_slice = binary_skeleton[im, :, :]
    loops_slice = loops_skel[im, :, :]
    mask_slice = input_img[im, :, :]
    contours, _ = cv2.findContours(mask_slice, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_image = np.zeros_like(skeleton_slice)
    contour_image = cv2.drawContours(contour_image, contours, -1, 255, thickness=5)
    contour_stack[im, :, :] = contour_image

    skeleton_new = np.copy(skeleton_slice)
    skeleton_new[(mask_slice == 0) & (skeleton_slice == 255)] = 0
    skeleton_new[(loops_slice == 255) & (contour_image == 255)] = 0

    filename_delooped = f"delooped_skeleton_{im}"
    filename_contour = f"contour_{im}"

    delooped_tif_writer.write(skeleton_new, description=filename_delooped, metadata=None)
    contour_tif_writer.write(contour_image, description=filename_contour, metadata=None)

delooped_tif_writer.close()
contour_tif_writer.close()

In [4]:
import tifffile as tf
four_um = tf.imread(r"F:\Megumi\DBOX-EQS1 Dropbox\Eqs1 Box03\Megumi-Cadisha\validation\voxel_size\GW13LU-B1and2\B1and2_part\4um\skeleton.tif")
seven_um = tf.imread(r"F:\Megumi\DBOX-EQS1 Dropbox\Eqs1 Box03\Megumi-Cadisha\validation\voxel_size\GW13LU-B1and2\B1and2_part\7um\skeleton.tif")
import napari
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(four_um, name = ('four_um'),  scale=[0.57, 0.57, 0.57])
viewer.add_image(seven_um, name='seven_um')

<Image layer 'seven_um' at 0x26d50e3b310>